In [2]:
###RAG Pipelines-Data Ingestion to Vector DB Pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11048\4064311150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
c:\Python\AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    #Find all pdf files recursively
    pdf_files =list(pdf_dir.glob("**/*.pdf"))
    print (f"found {len(pdf_files)} PDF files to process")
    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        except Exception as e:
            print(f" ❌Error {e}")
    print(f"\n all documents loaded")
    return all_documents

all_documents=process_all_pdfs("../data")



found 2 PDF files to process

 Processing: ai_rag_notes.pdf
loaded 1 pages

 Processing: python_notes.pdf
loaded 1 pages

 all documents loaded


In [7]:
all_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}, page_content='Ai Rag Notes\nArtificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can perform\ntasks associated with human intelligence, such as reasoning, learning, perception, and language\nunderstanding.\nMachine Learning\nMachine Learning is a subset of AI in which algorithms learn patterns from data and use those patterns to\nmake predictions or decisions.\nGenerative AI\nGenerative AI systems create new content such as text, images, audio, video, and code. Large language\

In [52]:
###Text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"{len(documents)} documents coverted into {len(split_docs)} chunks")
   # example of chunks
    if split_docs:
     print(f"\n Example chunks")
     print(f"Contents:{split_docs[0].page_content[:100]}...")
     print(f"Metadata:{split_docs[0].metadata}")

    return split_docs



In [9]:
chunks=split_documents(all_documents)
chunks

2 documents coverted into 3 chunks

 Example chunks
Contents:Ai Rag Notes
Artificial Intelligence
Artificial Intelligence (AI) is the field of computer science c...
Metadata:{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}, page_content='Ai Rag Notes\nArtificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can perform\ntasks associated with human intelligence, such as reasoning, learning, perception, and language\nunderstanding.\nMachine Learning\nMachine Learning is a subset of AI in which algorithms learn patterns from data and use those patterns to\nmake predictions or decisions.\nGenerative AI\nGenerative AI systems create new content such as text, images, audio, video, and code. Large language\

###embedding and VectorStoreDB

In [51]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Dict,Tuple,List,Any
from sklearn.metrics.pairwise import cosine_similarity

In [53]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Loaded embedding model successfully.Embedding dimensions {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise 
    def generate_embedding(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(texts)} texts ...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3107.96it/s]


Loaded embedding model successfully.Embedding dimensions 384


###VectorStore


In [57]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents_clean",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store {e}")
            raise
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding{len(documents)}documents to vector store")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"successfully added {len(documents)} to vector store")
            print(f"Total documents in collections:{self.collection.count()}")
        except Exception as e:
            print(f"failed to add to vector db:{e}")
            raise
vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents_clean
Existing documents in collection: 3


In [58]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-31T04:54:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-31T04:54:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\ai_rag_notes.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_rag_notes.pdf', 'file_type': 'pdf'}, page_content='Ai Rag Notes\nArtificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can perform\ntasks associated with human intelligence, such as reasoning, learning, perception, and language\nunderstanding.\nMachine Learning\nMachine Learning is a subset of AI in which algorithms learn patterns from data and use those patterns to\nmake predictions or decisions.\nGenerative AI\nGenerative AI systems create new content such as text, images, audio, video, and code. Large language\

In [59]:
###Convert text to embeddings
texts=[doc.page_content for doc in chunks]

###Generate the Embeddings
embeddings=embedding_manager.generate_embedding(texts)

###store in the vector database
vectorstore.add_documents(chunks,embeddings)


Generating embedding for 3 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.68it/s]

Generated embeddings with shape (3, 384)
Adding3documents to vector store
successfully added 3 to vector store
Total documents in collections:6


###Rag retriever pipeline


In [60]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    def retrieve(self,query:str,top_k:int =5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        print(f"Retrieving documents for query:'{query}'")
        print(f"Top K:{top_k}, Score threshold:{score_threshold}")

        query_embedding=self.embedding_manager.generate_embedding([query])[0]
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score = 1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print('no document found')
            return retrieved_docs
        except Exception as e:
            print(f"Error during Retrieval {e}")
            return[]
rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [61]:
rag_retriever

In [62]:
rag_retriever.retrieve("What is Generative AI?")

Retrieving documents for query:'What is Generative AI?'
Top K:5, Score threshold:0.0
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 60.52it/s]

Generated embeddings with shape (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_6616be6e_0',
  'content': 'Ai Rag Notes\nArtificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems that can perform\ntasks associated with human intelligence, such as reasoning, learning, perception, and language\nunderstanding.\nMachine Learning\nMachine Learning is a subset of AI in which algorithms learn patterns from data and use those patterns to\nmake predictions or decisions.\nGenerative AI\nGenerative AI systems create new content such as text, images, audio, video, and code. Large language\nmodels are commonly used for text generation.\nRetrieval-Augmented Generation\nRAG combines information retrieval with text generation. A RAG system retrieves relevant information from\nan external knowledge source and provides that information to a language model as context.\nRAG Pipeline\nA typical RAG pipeline loads documents, splits them into smaller chunks, creates embeddings for the chunks,',
  'metadata': {'pro

###Integration Vectordb context pipeline with LLM output

In [ ]:
###Simple RAG pipeline with groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key="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"
llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}
        Question:{query}
        Answer:"""
    response=llm.invoke([prompt.format(context=context,quert=query)])
    return response.content

In [64]:
answer=rag_simple("What is Machine Learning?",rag_retriever,llm)
print(answer)

Retrieving documents for query:'What is Machine Learning?'
Top K:3, Score threshold:0.0
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.56it/s]

Generated embeddings with shape (1, 384)
Retrieved 2 documents (after filtering)


Machine Learning is a subset of artificial intelligence in which algorithms automatically learn patterns from data and use those patterns to make predictions or decisions.


###Enhanced RAG Pipeline Features

In [66]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt)
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced(" What is Machine Learning?", rag_retriever, llm, top_k=3, min_score=0.3, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query:' What is Machine Learning?'
Top K:3, Score threshold:0.3
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.23it/s]

Generated embeddings with shape (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [67]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Machine Learning?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query:'what is Machine Learning?'
Top K:3, Score threshold:0.1
Generating embedding for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 68.55it/s]

Generated embeddings with shape (1, 384)
Retrieved 0 documents (after filtering)



Final Answer: No relevant context found.
Summary: The reply indicates that there is no pertinent information available. In other words, the necessary context is missing.
History: {'question': 'what is Machine Learning?', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'The reply indicates that there is no pertinent information available. In other words, the necessary context is missing.'}
